# Initialize

## Load Libraries

In [ ]:
import matplotlib
import random
import numpy as np
import matplotlib.pyplot as plt

## Load Data

In [ ]:
data  = np.transpose(np.loadtxt('g-Factor.txt', skiprows=1, delimiter = '\t'))

# This is to skip the last few points of the data set
padding = -100
t = data[0,:padding]
gx = data[1,:padding]
gy = data[2,:padding]
gz = data[3,:padding]

In [ ]:
fs = 12
fs_x = 14
fs_y = 9

# Expected value for g
Eidgenössisches Institut für Metrologie METAS:
https://www.metas.ch/metas/de/home/dok/gravitationszonen.html

In [ ]:
g_theo = 9.806 # N/kg

# Main

In [ ]:
# Get size of data set
n = gz.shape[0]

print('Number of data points : {:d}'.format(n))

dt = (t[-1] - t[0]) / n 

print('Time per data point : {:0.2f} ms'.format(dt * 1000))

## Plot Data

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.plot(t, gz, 'b.', label='Data')

# Labels and legend
ax1.set_xlabel(r'$t$ (s)')
ax1.set_ylabel(r'$g_\mathrm{z}$ (N/kg)')
ax1.legend()
plt.show()

## Discretize data by sorting it into bins

In [ ]:
# Number of bins and bin size

gz_max = np.max(gz)
gz_min = np.min(gz)


#bin_size = 0.01
#bin_size = 0.005
bin_size = 0.002

n_bins = int((gz_max - gz_min) / bin_size) + 1

bins = np.linspace(gz_min, gz_min+bin_size*(n_bins-1), n_bins)

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.plot(t, gz, 'b.', label='Data', zorder = 1)
for b in bins:
    ax1.hlines(b, 0, t[-1], 'r', zorder = 2)

# Labels and legend
ax1.set_xlabel(r'$t$ (s)')
ax1.set_ylabel(r'$g_\mathrm{z}$ (N/kg)')
ax1.legend()
plt.show()

In [ ]:
# Initialize array holding number of data  points (occurrence) per bin
occurrence = np.zeros(n_bins)

# Calculate the offset of the occurrence-array index with respect to the floor division of the data
offset = np.min(bins)//bin_size + 1

# Iterate through all data and find the index of the occurrence-array into which the data falls.
# Then increment this array value by one. 
for gz_i in gz:
    index = int(gz_i//bin_size-offset)
    occurrence[index] = occurrence[index] + 1

## Plot Histogram

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.bar(bins, occurrence, width = bin_size)
ax1.vlines(g_theo, 0, np.max(occurrence), 'r', label='Theoretical value')

# Labels and legend
ax1.set_xlabel(r'$g_\mathrm{z}$ (N/kg)')
ax1.set_ylabel('Occurrences')
ax1.legend()
plt.show()

## Calculate Mean and Variance

In [ ]:
mean = np.sum(gz)/n
var = np.sum((gz-mean)**2)/(n-1)

In [ ]:
print('Mean : {:0.2f} N/kg, Standard deviation (std) : {:0.2f} N/kg'.format(mean, np.sqrt(var)))

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.bar(bins, occurrence, width = bin_size)
ax1.vlines(g_theo, 0, np.max(occurrence), 'r', label='Theoretical value')
ax1.vlines(mean, 0, np.max(occurrence), 'g', label='Mean')
ax1.vlines(mean + np.sqrt(var), 0, np.max(occurrence), 'g', linestyles='dashed', label='Mean +/- std')
ax1.vlines(mean - np.sqrt(var), 0, np.max(occurrence), 'g', linestyles='dashed')

# Labels and legend
ax1.set_xlabel(r'$g_\mathrm{z}$ (N/kg)')
ax1.set_ylabel('Occurrences')
ax1.legend()
plt.show()

We have only one Datapoint within our 1-$\sigma$-interval. We should increase the number of bins, and decrease the bin size accordingly.

<b>Question:</b> Are the data points normally distributed?

In [ ]:
# Calculate the fraction of the data point within a certain interval around the mean.

# Sum of the data points in the interval
n_sigma = 0
# Width of the interval in terms of the standard deviation
width = 1
# iterate through all bins and sum their values if they lie in the specified interval
i = 0
for b in bins:
    if (b >= (mean - np.sqrt(var) * width)) and (b <= (mean + np.sqrt(var) * width)):
        n_sigma = n_sigma + occurrence[i]
    i = i + 1
    
print('Occurrences within the {:d}-sigma-interval: {:0.1f}%'.format(width,n_sigma/n*100))

## Probability Distribution

In [ ]:
# Probability normalization factor (the favorable over the possible)
A = n

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.bar(bins, occurrence/A, width = bin_size)
ax1.vlines(g_theo, 0, np.max(occurrence/A), 'r')
ax1.vlines(mean, 0, np.max(occurrence/A), 'g')
ax1.vlines(mean + np.sqrt(var), 0, np.max(occurrence/A), 'g', linestyles='dashed')
ax1.vlines(mean - np.sqrt(var), 0, np.max(occurrence/A), 'g', linestyles='dashed')

# Labels and legend
ax1.set_xlabel(r'$g_\mathrm{z}$ (N/kg)')
ax1.set_ylabel('Probability')
plt.show()

## Deviation from Theory Value

In [ ]:
d = (g_theo - mean) / np.sqrt(var)

In [ ]:
print('Deviation of the mean from the theory value: {:0.1f} sigma'.format(d))

# Standard Error

## Plot data point by point

In [ ]:
# Number of points to be plotted

#n_max = 1
#n_max = 2
#n_max = 3
n_max = 10
#n_max = 100

mean_n = np.sum(gz[0:n_max])/n_max

if n_max > 1:
    std_n = np.sqrt(np.sum((gz[0:n_max] - mean_n)**2) / (n_max - 1))

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.plot(t[0:n_max], gz[0:n_max], 'b.', label='Data')

ax1.hlines(mean, t[0] - 0.001, t[n_max - 1] + 0.001, colors='0.3', label='Theoretical value')

ax1.hlines(mean_n, t[0] - 0.001, t[n_max - 1] + 0.001, colors='0.5', linestyles='dashed', label='Mean')

if n_max > 1:
    ax1.hlines(mean_n + std_n, t[0] - 0.001, t[n_max - 1] + 0.001, colors='0.5', linestyles='dotted', label='Mean +/- std')
    ax1.hlines(mean_n - std_n, t[0] - 0.001, t[n_max - 1] + 0.001, colors='0.5', linestyles='dotted')
# Labels and legend
ax1.set_xlabel(r'$t$ (s)')
ax1.set_ylabel(r'$g_\mathrm{z}$ (N/kg)')
ax1.legend()
plt.show()

## Plot mean and standard deviation as function of the number of used data points

In [ ]:
mean_n = np.zeros(n)
std_n = np.zeros(n)

nn = np.linspace(1, n, n, dtype=np.int16)

for i in nn:
    mean_n[i-1] = np.sum(gz[0:i]) / i
    if i > 1:
       std_n[i-1] = np.sqrt(np.sum((gz[0:i] - mean_n[i - 1])**2) / (i - 1)) 

In [ ]:
# Simple Figure
# =============
# cm to inch
cm = 0.393701

# Define Font Size
plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, 1.5 * fs_y*cm))
ax1 = fig.add_subplot(2, 1, 1)

# Plot Data
ax1.plot(nn, mean_n, 'b')
ax1.set_xscale('log')

ax1.set_xticklabels([])

ax2 = fig.add_subplot(2, 1, 2)

# Plot Data
ax2.plot(nn, std_n, 'b')
ax2.set_xscale('log')
# Labels and legend
ax2.set_xlabel(r'$t$ (s)', fontsize=20)
ax1.set_ylabel(r'$\bar{g}_\mathrm{z}$ (N/kg)')

ax2.set_ylabel(r'std$(g)_\mathrm{z}$ (N/kg)')

plt.show()

# Calculate the error of the mean

In [ ]:
std_mean_n = np.zeros(n)

for i in nn[1:]:
    std_mean_n[i - 1] = std_n[i-1] / np.sqrt(i-1)

In [ ]:
# Simple Figure
# =============
# cm to inch
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, 1.5 * fs_y*cm))
ax1 = fig.add_subplot(2, 1, 1)

# Plot Data
ax1.plot(nn, mean_n, 'b', label='Mean')

ax1.plot(nn[1:], mean_n[1:] + std_mean_n[1:], 'r', label='Mean +/- std')
ax1.plot(nn[1:], mean_n[1:] - std_mean_n[1:], 'r')

ax1.set_xscale('log')

ax1.set_xticklabels([])

ax2.legend()

ax2 = fig.add_subplot(2, 1, 2)

# Plot Data
ax2.plot(nn[1:], std_n[1:], 'b', label='Std', zorder = 1)
ax2.plot(nn[1:], std_mean_n[1:], 'r', label='Error of the mean', zorder = 1)

ax2.set_xscale('log')
# Labels and legend
ax2.set_xlabel(r'$t$ (s)')
ax1.set_ylabel(r'$\bar{g}_\mathrm{z}$ (N/kg)')

ax2.set_ylabel(r'$\Delta g_\mathrm{z}$ (N/kg)')

ax2.legend()
plt.show()

# More Statistics on the Error of the Mean

In [ ]:
# Number of measurements
m = 200
# Number of data points per measurement
n = 200

# Simulation of g-factor measurement
mu = mean
sigma = np.sqrt(var)

# 2D array of m x n measurements
x = np.random.normal(mu, sigma, size = (n, m))

# 1D array indexing the measurement number
nn = np.linspace(1, n, n, dtype='int16')

In [ ]:
# Plot data from different measurements
m_plot_a = 0
m_plot_b = 1

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.plot(nn * dt, x[:, m_plot_a], 'b.', label='Measurement a')
ax1.plot(nn * dt, x[:, m_plot_b], 'r.', label='Measurement b')


# Labels and legend
ax1.set_xlabel(r'$t$ (s)')
ax1.set_ylabel(r'$g_\mathrm{z}$ (N/kg)')
ax1.legend()
plt.show()

## Calculate mean, standard deviation and error of the mean

In [ ]:
# Define arrays for the calculated results
# Arrays holding the mean and std and std of the mean values for EACH measurement m calculated for the data values 1 -> n_j 
mean_n = np.zeros((n, m))
std_n = np.zeros((n, m))
std_mean_n = np.zeros((n, m))

# Arrays holding the mean, std of the mean and mean of the std calculated for ALL measurements m for data values 1 -> n_j
mean_m = np.zeros(n)
std_mean_m = np.zeros(n)
mean_std_m = np.zeros(n)

# Cycle through all n points of the measurements j: index of the data point, n_j: number of data points 
for j, n_j in enumerate(nn):
    # Cycle through all m measurements i: index of the measurement, m_i: number of measurements
    for i, m_i in enumerate(nn):
        # Calculate the mean for the data points 1 - n_j for each of the m measurements
        mean_n[j, i] = np.sum(x[0:n_j, i]) / n_j
        # (we calculate the stds only if more than one data point is considered)
        if n_j>1:
            # Calculate the std for the data points 1 - n_j for each of the m measurements 
            std_n[j, i] = np.sqrt(np.sum((x[0:n_j, i] - mean_n[j, i])**2) / (n_j - 1))
            # Calculate the standard error of the means calculated before using the expression from the lecture
            std_mean_n[j, i] = std_n[j, i] / np.sqrt(n_j)
    # Statistics of the m measurements
    # Calculate the mean of the means of all m measurements
    mean_m[j] =  np.sum(mean_n[j, :]) / m_i
    # Calculate the std of the means of all m measurements (this should be equivalent to the standard error)
    std_mean_m[j] =  np.sqrt(np.sum((mean_n[j, :] - mean_m[j])**2) / (m_i - 1))
    # Calculate the mean of the stds of all m measurements
    mean_std_m[j] =  np.sum(std_n[j, :]) / m_i

### Plot data, mean and standard deviation

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.plot(nn * dt,  mean_n[:, m_plot_a], 'b')

ax1.plot(nn * dt,  mean_n[:, m_plot_a] + std_n[:, m_plot_a], 'b', linestyle='dashed')
ax1.plot(nn * dt,  mean_n[:, m_plot_a] - std_n[:, m_plot_a], 'b', linestyle='dashed')

ax1.plot(nn * dt,  mean_n[:, m_plot_b], 'r', label='Data')

ax1.plot(nn * dt,  mean_n[:, m_plot_b] + std_n[:, m_plot_b], 'r', linestyle='dashed')
ax1.plot(nn * dt,  mean_n[:, m_plot_b] - std_n[:, m_plot_b], 'r', linestyle='dashed')

ax1.plot(nn * dt, x[:, m_plot_a], 'b.')
ax1.plot(nn * dt, x[:, m_plot_b], 'r.')


# Labels and legend
ax1.set_xlabel(r'$t$ (s)')
ax1.set_ylabel(r'$g_\mathrm{z}$ (N/kg)')

plt.show()

### Plot the means calculated for different number of data points (1, 10 and all) as function of the measurements

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data

ax1.plot(nn,  mean_n[0, :], 'b.', label='$n_j = 1$')
ax1.plot(nn,  mean_n[10, :], 'r.', label='$n_j = 10$')
ax1.plot(nn,  mean_n[-1, :], 'g.', label='All data points')



# Labels and legend
ax1.set_xlabel(r'Measurements $m_j$')
ax1.set_ylabel(r'$g_\mathrm{z}$ (N/kg)')
ax1.legend()
plt.show()

### Calculate histogram of the mean for different number of data points

In [ ]:
# Number of bins and bin size

gz_max_0 = np.max(mean_n[1,:])
gz_min_0 = np.min(mean_n[1,:])

bin_size_0 = 0.002

n_bins_0 = int((gz_max_0 - gz_min_0) / bin_size_0) + 1

bins_0 = np.linspace(gz_min_0, gz_min_0 + bin_size_0 * (n_bins_0 - 1), n_bins_0)

# Initialize array holding number of data  points (occurrence) per bin
occurrence_0 = np.zeros(n_bins_0)

# Calculate the offset of the occurrence-array index with respect to the floor division of the data
offset = np.min(bins_0)//bin_size_0 + 1

# Iterate through all data and find the index of the occurrence-array into which the data falls.
# Then increment this array value by one. 
for gz_i in mean_n[1,:]:
    index = int(gz_i//bin_size_0-offset)
    occurrence_0[index] = occurrence_0[index] + 1

In [ ]:
# Number of bins and bin size

gz_max_1 = np.max(mean_n[10,:])
gz_min_1 = np.min(mean_n[10,:])

bin_size_1 = 0.0005

n_bins_1 = int((gz_max_1 - gz_min_1) / bin_size_1) + 1

bins_1 = np.linspace(gz_min_1, gz_min_1 + bin_size_1 * (n_bins_1 - 1), n_bins_1)

# Initialize array holding number of data  points (occurrence) per bin
occurrence_1 = np.zeros(n_bins_1)

# Calculate the offset of the occurrence-array index with respect to the floor division of the data
offset = np.min(bins_1)//bin_size_1 + 1

# Iterate through all data and find the index of the occurrence-array into which the data falls.
# Then increment this array value by one. 
for gz_i in mean_n[10,:]:
    index = int(gz_i//bin_size_1-offset)
    occurrence_1[index] = occurrence_1[index] + 1

In [ ]:
# Number of bins and bin size

gz_max_2 = np.max(mean_n[-1,:])
gz_min_2 = np.min(mean_n[-1,:])

bin_size_2 = 0.0002

n_bins_2 = int((gz_max_2 - gz_min_2) / bin_size_2) + 1

bins_2 = np.linspace(gz_min_2, gz_min_2 + bin_size_2 * (n_bins_2 - 1), n_bins_2)

# Initialize array holding number of data  points (occurrence) per bin
occurrence_2 = np.zeros(n_bins_2)

# Calculate the offset of the occurrence-array index with respect to the floor division of the data
offset = np.min(bins_2)//bin_size_2 + 1

# Iterate through all data and find the index of the occurrence-array into which the data falls.
# Then increment this array value by one. 
for gz_i in mean_n[-1,:]:
    index = int(gz_i//bin_size_2-offset)
    occurrence_2[index] = occurrence_2[index] + 1

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.bar(bins_0, occurrence_0, width = bin_size, label='$n_j = 1$')
ax1.bar(bins_1, occurrence_1, width = bin_size * 0.8, label='$n_j = 10$')
ax1.bar(bins_2, occurrence_2, width = bin_size * 0.4, label='all data points')

# Labels and legend
ax1.set_xlabel(r'$g_\mathrm{z}$ (N/kg)')
ax1.set_ylabel('Occurrences')

ax1.legend()

plt.show()

### Plot the evolution of the mean, std, and std of the mean (for all measurements) based for data points $1 \rightarrow n_j$

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.plot(nn,  mean_m, 'b.', label='Mean')


# Labels and legend
ax1.set_xlabel(r'$t$ (s)')
ax1.set_ylabel(r'$\bar{g}_\mathrm{z}$ (N/kg)')
ax1.legend()
plt.show()

In [ ]:
cm = 0.393701

plt.rcParams.update({'font.size': fs})
plt.rcParams.update({'axes.labelsize': fs})

fig = plt.figure(figsize=(fs_x*cm, fs_y*cm))
ax1 = fig.add_subplot(1, 1, 1)

# Plot data
ax1.plot(nn,  std_mean_m, 'r', label='Std of the mean')
ax1.plot(nn,  mean_std_m, 'b', label='Mean of the std')
ax1.plot(nn,  mean_std_m / np.sqrt(nn), 'g', label='Error of the mean')

ax1.set_xscale('log')

# Labels and legend
ax1.set_xlabel(r'$t$ (s)')
ax1.set_ylabel(r'$\Delta g_\mathrm{z}$ (N/kg)')
ax1.legend()
plt.show()